# MONAI + RankSEG 医疗分割后处理 Demo（Colab 默认，中文）

本 Notebook 目标：
1. 在 **Google Colab** 上默认可运行；
2. 用 **uv** 管理 Python 包；
3. 从你当前 fork 的源码安装 MONAI（包含最新 RankSEG 集成）；
4. 完成一套 **算法与数据 pipeline 全流程检查**（shape、数值范围、标签合法性、指标对比、可视化）。


## 0) Colab 环境准备（必须先执行）

> 下面命令是 Colab 风格（`!`）。如果你在本地 Jupyter，请改成终端命令。


In [ ]:
# ===== Colab: 安装 uv =====
!pip -q install uv

# ===== Colab: 克隆你的 fork（请把 URL 改成你的仓库） =====
REPO_URL = "https://github.com/<your-username>/MONAI.git"  # <- 改成你的 fork URL
BRANCH = "work"  # <- 改成你要验证的分支

!rm -rf /content/MONAI
!git clone --branch {BRANCH} {REPO_URL} /content/MONAI

# ===== 用 uv 安装依赖（系统环境） =====
!uv pip install --system -U pip setuptools wheel
!uv pip install --system -e /content/MONAI
!uv pip install --system rankseg matplotlib numpy pandas

print("环境安装完成。")


## 1) 导入依赖并设置随机种子


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from monai.transforms import Activations, AsDiscrete, RankSeg

torch.manual_seed(2026)
np.random.seed(2026)

print("torch:", torch.__version__)


## 2) 构建合成医疗数据（CT-like 器官/病灶）

标签定义：
- `0`: 背景
- `1`: 器官
- `2`: 病灶


In [ ]:
def make_case(h=128, w=128):
    yy, xx = np.mgrid[0:h, 0:w]

    # 器官：椭圆区域
    cx, cy = np.random.randint(45, 85), np.random.randint(45, 85)
    rx, ry = np.random.randint(20, 32), np.random.randint(18, 28)
    organ = (((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2) <= 1.0

    # 病灶：器官内部的小圆形区域
    lx, ly = cx + np.random.randint(-8, 8), cy + np.random.randint(-8, 8)
    lr = np.random.randint(6, 11)
    lesion = ((xx - lx) ** 2 + (yy - ly) ** 2) <= lr**2
    lesion = lesion & organ

    label = np.zeros((h, w), dtype=np.int64)
    label[organ] = 1
    label[lesion] = 2

    # CT-like 灰度（仅用于可视化）
    image = np.random.normal(loc=-100, scale=30, size=(h, w)).astype(np.float32)
    image[organ] = np.random.normal(loc=60, scale=25, size=organ.sum()).astype(np.float32)
    image[lesion] = np.random.normal(loc=130, scale=20, size=lesion.sum()).astype(np.float32)

    # 模拟模型 logits：GT 偏置 + 噪声 + 边界扰动
    logits = np.random.normal(0, 0.9, size=(3, h, w)).astype(np.float32)
    logits[0] += (label == 0) * 1.6
    logits[1] += (label == 1) * 1.6
    logits[2] += (label == 2) * 1.6

    boundary = np.zeros_like(label, dtype=bool)
    boundary[1:, :] |= label[1:, :] != label[:-1, :]
    boundary[:, 1:] |= label[:, 1:] != label[:, :-1]
    logits[:, boundary] += np.random.normal(0, 1.0, size=(3, boundary.sum())).astype(np.float32)

    return image, label, logits


def multiclass_dice(pred, gt, cls):
    p = pred == cls
    g = gt == cls
    inter = (p & g).sum()
    denom = p.sum() + g.sum()
    return 1.0 if denom == 0 else (2.0 * inter) / denom


## 3) 数据 pipeline 全检查（Data QA）


In [ ]:
num_cases = 10
images, labels, logits = [], [], []
for _ in range(num_cases):
    img, lab, logit = make_case()
    images.append(img)
    labels.append(lab)
    logits.append(logit)

images_t = torch.tensor(np.stack(images), dtype=torch.float32)       # [B, H, W]
labels_t = torch.tensor(np.stack(labels), dtype=torch.long)          # [B, H, W]
logits_t = torch.tensor(np.stack(logits), dtype=torch.float32)       # [B, C, H, W]

# ===== 数据质量检查 =====
assert images_t.ndim == 3, f"images_t shape 错误: {images_t.shape}"
assert labels_t.ndim == 3, f"labels_t shape 错误: {labels_t.shape}"
assert logits_t.ndim == 4 and logits_t.shape[1] == 3, f"logits_t shape 错误: {logits_t.shape}"

u = torch.unique(labels_t)
assert set(u.tolist()).issubset({0,1,2}), f"标签值越界: {u.tolist()}"

assert torch.isfinite(images_t).all(), "images_t 有 NaN/Inf"
assert torch.isfinite(logits_t).all(), "logits_t 有 NaN/Inf"

print("Data QA 通过")
print("images_t:", tuple(images_t.shape), "labels_t:", tuple(labels_t.shape), "logits_t:", tuple(logits_t.shape))


## 4) 算法 pipeline：Baseline vs RankSEG

- Baseline: `softmax + argmax`
- RankSEG: `softmax + RankSeg(metric='dice')`


In [ ]:
act = Activations(softmax=True, dim=1)
base_disc = AsDiscrete(argmax=True, dim=1, keepdim=False)
rankseg_disc = RankSeg(metric='dice', mode='multiclass', output_mode='multiclass', solver='RMA')

probs_t = act(logits_t)
pred_base = base_disc(probs_t).long()    # [B, H, W]
pred_rank = rankseg_disc(probs_t).long() # [B, H, W]

# ===== 算法输出检查 =====
assert probs_t.shape == logits_t.shape, "probs shape 不匹配"
assert pred_base.shape == labels_t.shape, "baseline shape 不匹配"
assert pred_rank.shape == labels_t.shape, "rankseg shape 不匹配"

# softmax 概率和应接近 1
psum = probs_t.sum(dim=1)
assert torch.allclose(psum, torch.ones_like(psum), atol=1e-4), "softmax 概率和异常"

# 离散标签范围检查
assert set(torch.unique(pred_base).tolist()).issubset({0,1,2}), "baseline 标签越界"
assert set(torch.unique(pred_rank).tolist()).issubset({0,1,2}), "rankseg 标签越界"

print("Algorithm QA 通过")
print("probs:", tuple(probs_t.shape), "base:", tuple(pred_base.shape), "rank:", tuple(pred_rank.shape))


## 5) Pair-by-pair 指标比较（逐病例）

每个病例计算器官类/病灶类 Dice，并统计 `RankSEG - Baseline` 的提升。


In [ ]:
rows = []
for i in range(num_cases):
    gt = labels_t[i].cpu().numpy()
    b = pred_base[i].cpu().numpy()
    r = pred_rank[i].cpu().numpy()

    b_d1 = multiclass_dice(b, gt, 1)
    b_d2 = multiclass_dice(b, gt, 2)
    r_d1 = multiclass_dice(r, gt, 1)
    r_d2 = multiclass_dice(r, gt, 2)

    rows.append({
        "case": i,
        "baseline_dice_organ": b_d1,
        "baseline_dice_lesion": b_d2,
        "rankseg_dice_organ": r_d1,
        "rankseg_dice_lesion": r_d2,
        "baseline_mean_dice": (b_d1 + b_d2) / 2,
        "rankseg_mean_dice": (r_d1 + r_d2) / 2,
        "delta_mean_dice": ((r_d1 + r_d2) - (b_d1 + b_d2)) / 2,
    })

result_df = pd.DataFrame(rows)
result_df = result_df.sort_values("delta_mean_dice", ascending=False).reset_index(drop=True)

# 指标范围检查
for col in ["baseline_dice_organ", "baseline_dice_lesion", "rankseg_dice_organ", "rankseg_dice_lesion", "baseline_mean_dice", "rankseg_mean_dice"]:
    assert ((result_df[col] >= 0) & (result_df[col] <= 1)).all(), f"{col} 超出 [0,1]"

print("Metric QA 通过")
print(result_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("
平均提升 delta_mean_dice:", f"{result_df['delta_mean_dice'].mean():+.4f}")


## 6) Case-by-case 可视化（成对展示）

每行一个病例：`Image | GT | Baseline | RankSEG`。


In [ ]:
n_show = min(8, num_cases)
fig, axes = plt.subplots(n_show, 4, figsize=(14, 3 * n_show))
if n_show == 1:
    axes = np.expand_dims(axes, axis=0)

# 用原 case 顺序展示，和指标表可以相互对照
for i in range(n_show):
    axes[i, 0].imshow(images_t[i].cpu().numpy(), cmap="gray")
    axes[i, 0].set_title(f"Case {i} - Image")

    axes[i, 1].imshow(labels_t[i].cpu().numpy(), vmin=0, vmax=2, cmap="viridis")
    axes[i, 1].set_title("GT")

    axes[i, 2].imshow(pred_base[i].cpu().numpy(), vmin=0, vmax=2, cmap="viridis")
    axes[i, 2].set_title("Baseline (argmax)")

    axes[i, 3].imshow(pred_rank[i].cpu().numpy(), vmin=0, vmax=2, cmap="viridis")
    axes[i, 3].set_title("RankSEG")

    for j in range(4):
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()


## 7) 真实项目最小侵入接入建议

在真实 MONAI 推理配置里，直接把后处理链从：
- `Activationsd -> AsDiscreted(argmax=True)`

替换为：
- `Activationsd -> RankSegd(keys='pred', metric='dice', mode='multiclass', output_mode='multiclass')`

即可完成最小侵入集成，并保留后续 `SaveImaged`/metrics handlers。
